# Dog vs Cat Classification

**Computer Vision Project** — Classify images as dog or cat.

Model: PyTorch CNN  
Dataset: Synthetic sample (auto-generated) or Kaggle dogs-vs-cats  
Target: 0=cat, 1=dog

## 2 · Project Overview

We build a **CNN** to classify images of dogs and cats. This is one of the most popular introductory CV classification tasks. We use a lightweight CNN architecture that can train quickly even without GPU.

For production, a pretrained DINOv3 or ConvNeXt V2 backbone would be preferred. This notebook focuses on CNN fundamentals.

## 3 · Learning Objectives

1. Build a CNN for binary image classification.
2. Use torchvision transforms for data augmentation.
3. Train with proper train/val/test splits.
4. Evaluate with accuracy, F1, and confusion matrix.
5. Understand overfitting on small image datasets.

## 4 · Problem Statement

Given an image, classify whether it contains a **dog** or a **cat**. Binary image classification.

## 5 · Why This Project Matters

- Dog vs Cat is the canonical binary image classification benchmark.
- Teaches CNN architecture design, data augmentation, and transfer learning concepts.
- Real-world applications: pet detection, animal shelter automation, wildlife monitoring.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Task** | Binary image classification |
| **Classes** | 2 (cat=0, dog=1) |
| **Image format** | RGB |
| **Source** | Synthetic generation for reproducibility |

We generate a synthetic dataset with simple patterns to ensure the notebook runs without external downloads. For real performance, use the Kaggle dogs-vs-cats dataset.

## 7 · Dataset Source and License Notes

- **Synthetic data** generated in-notebook for reproducibility.
- For real data: Kaggle dogs-vs-cats dataset (Microsoft Research license).
- Production models should use real image datasets.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_install("torch")
_install("torchvision")
print("All packages ready.")

## 9 · Imports

In [ ]:
import warnings, os, json
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 10 · Configuration / Constants

In [ ]:
IMG_SIZE = 32
NUM_CHANNELS = 3
NUM_CLASSES = 2
BATCH_SIZE = 32
EPOCHS = 10
LR = 0.001
N_SAMPLES = 2000  # synthetic samples
print(f"Image: {IMG_SIZE}x{IMG_SIZE}x{NUM_CHANNELS}, Samples: {N_SAMPLES}, Epochs: {EPOCHS}")

## 11 · Dataset Generation

We create synthetic images: **cats** have horizontal stripe patterns, **dogs** have vertical stripe patterns. This gives the CNN learnable features while keeping the notebook self-contained and fast.

In [ ]:
def generate_synthetic_data(n_samples, img_size, seed=42):
    rng = np.random.RandomState(seed)
    images = []
    labels = []
    for i in range(n_samples):
        img = rng.rand(3, img_size, img_size).astype(np.float32) * 0.3
        label = i % 2  # alternating cat/dog
        if label == 0:  # cat: horizontal stripes
            for r in range(0, img_size, 4):
                img[:, r:r+2, :] += 0.5
        else:  # dog: vertical stripes
            for c in range(0, img_size, 4):
                img[:, :, c:c+2] += 0.5
        img = np.clip(img, 0, 1)
        images.append(img)
        labels.append(label)
    return np.array(images), np.array(labels)

X_all, y_all = generate_synthetic_data(N_SAMPLES, IMG_SIZE, SEED)
X_tensor = torch.FloatTensor(X_all)
y_tensor = torch.LongTensor(y_all)
print(f"Generated: {X_tensor.shape}, Labels: {y_tensor.shape}")
print(f"Class distribution: {np.bincount(y_all)}")

## 12 · Data Validation Checks

In [ ]:
print(f"Image range: [{X_tensor.min():.3f}, {X_tensor.max():.3f}]")
print(f"Unique labels: {sorted(y_tensor.unique().tolist())}")
print(f"Class 0 (cat): {(y_tensor == 0).sum().item()}")
print(f"Class 1 (dog): {(y_tensor == 1).sum().item()}")

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i in range(5):
    axes[0, i].imshow(X_all[i*2].transpose(1,2,0))
    axes[0, i].set_title(f"Cat ({y_all[i*2]})")
    axes[0, i].axis("off")
    axes[1, i].imshow(X_all[i*2+1].transpose(1,2,0))
    axes[1, i].set_title(f"Dog ({y_all[i*2+1]})")
    axes[1, i].axis("off")
plt.suptitle("Sample Images")
plt.tight_layout()
plt.show()

## 13 · Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Cat (0)", "Dog (1)"], np.bincount(y_all), color=["steelblue", "coral"])
ax.set_title("Class Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_class_distribution.png"), dpi=100)
plt.show()

# Channel statistics
for ch, name in enumerate(["Red", "Green", "Blue"]):
    print(f"{name}: mean={X_all[:, ch].mean():.3f}, std={X_all[:, ch].std():.3f}")

## 14 · Target Analysis

Perfectly balanced (50/50). No class weighting needed.

## 15 · Train/Validation/Test Split Strategy

In [ ]:
dataset = TensorDataset(X_tensor, y_tensor)
n_total = len(dataset)
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test],
                                          generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {n_train}, Val: {n_val}, Test: {n_test}")

## 16 · Preprocessing Strategy

Images are already normalized to [0, 1]. For real images, we'd apply ImageNet normalization and data augmentation.

## 17 · Feature Engineering

CNN layers automatically learn hierarchical features from raw pixels.

## 18 · Baseline Model — CNN

In [ ]:
class DogCatCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, NUM_CLASSES),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = DogCatCNN().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

## 19 · LazyPredict Benchmark

**Not applicable for computer vision / CNN tasks.** LazyPredict is designed for tabular data only.

## 20 · FLAML AutoML Run

**Not applicable for computer vision / CNN tasks.** FLAML is designed for tabular AutoML.

## 21 · Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(EPOCHS):
    model.train()
    rl, rc, rt = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        rl += loss.item() * xb.size(0)
        rc += (out.argmax(1) == yb).sum().item()
        rt += xb.size(0)
    train_losses.append(rl / rt)
    train_accs.append(rc / rt)

    model.eval()
    vl, vc, vt = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            vl += loss.item() * xb.size(0)
            vc += (out.argmax(1) == yb).sum().item()
            vt += xb.size(0)
    val_losses.append(vl / vt)
    val_accs.append(vc / vt)

    print(f"Epoch {epoch+1}/{EPOCHS} — Train: Loss={train_losses[-1]:.4f} Acc={train_accs[-1]:.4f} | Val: Loss={val_losses[-1]:.4f} Acc={val_accs[-1]:.4f}")

## 22 · Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses, label="Train"); ax1.plot(val_losses, label="Val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss"); ax1.legend()
ax2.plot(train_accs, label="Train"); ax2.plot(val_accs, label="Val")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.set_title("Accuracy"); ax2.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "training_curves.png"), dpi=100)
plt.show()

## 23 · Final Evaluation on Test Set

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        out = model(xb)
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_labels.extend(yb.tolist())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")
print(f"Test Accuracy: {acc:.4f}")
print(f"Test F1: {f1:.4f}")

## 24 · Error Analysis

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Cat", "Dog"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues")
ax.set_title("Confusion Matrix — Dog vs Cat CNN")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=100)
plt.show()

print(classification_report(all_labels, all_preds, target_names=["Cat", "Dog"]))

wrong = np.where(all_preds != all_labels)[0]
print(f"Errors: {len(wrong)} / {len(all_labels)} ({len(wrong)/len(all_labels)*100:.1f}%)")

## 25 · Interpretation and Business Insight

- The CNN easily distinguishes the synthetic stripe patterns, achieving near-perfect accuracy.
- On real dog/cat images, a pretrained backbone (DINOv3, ConvNeXt V2) would be needed for >95% accuracy.
- For production, consider model distillation for mobile deployment.

## 26 · Limitations

- Synthetic data — real images are much harder.
- No data augmentation applied.
- Small CNN — real tasks need deeper architectures or transfer learning.

## 27 · How to Improve

1. Use real Kaggle dogs-vs-cats dataset.
2. Apply transfer learning with pretrained models.
3. Add data augmentation.
4. Try larger architectures.

## 28 · Production Considerations

- ONNX export for deployment.
- Input validation and confidence thresholding.
- Monitor for distribution shift.

## 29 · Common Mistakes

1. No data augmentation on small datasets.
2. No validation set.
3. Not using pretrained models for real images.
4. Overfitting without regularization.

## 30 · Mini Challenge

1. Add RandomHorizontalFlip and RandomRotation.
2. Try a deeper network with residual connections.
3. Use real images from Kaggle.

## 31 · Final Summary

In [ ]:
results = {"CNN": {"accuracy": acc, "f1": f1}}
metrics = {"best_model": "CNN", "best_accuracy": acc, "best_f1": f1, "epochs": EPOCHS}

torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pth"))
with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("Model + metrics saved.")
print(json.dumps(metrics, indent=2))
print("\n✅ Dog vs Cat Classification notebook complete.")